# <mark style="display:block; background:#d1c4e9; color:#1a1a1a; padding:6px 12px; border-radius:4px">10/2(금) 오전 · 웹훅 · 터미널 — 실습</mark>

나흘 동안 우리는 **보내는 쪽**이었습니다. 로그를 읽고, 룰로 고르고, 밖에 물어봤습니다.

그런데 옆 팀에서 요청이 왔습니다. **「alert(경보)가 발생하는 즉시 보낼 테니 받을 주소를 하나 만들어 달라」.**
우리는 지금까지 요청을 받아 본 적이 없습니다. **요청을 받으려면 서버가 있어야 합니다.**

오늘 오전의 도착점은 **`webhook_server.py`** 와 **`test_webhook.sh`** 입니다.


### 아침 과제 확인 셀

아침 과제 4번에서 이 셀을 실행합니다. `준비 완료` 가 나오면 오늘 수업 준비가 끝난 것입니다.

In [ ]:
import flask
import requests
import schedule

print("준비 완료")

## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">시작하기</mark>

### 0.1 맨 먼저 · 노트북과 터미널의 위치 맞추기

이 노트북은 `security-agent-toolkit` 의 **`agent_core` 폴더**에 있어야 합니다. 오늘 만드는 파일은 모두 이 폴더에 만들어집니다.

VS Code 터미널(Git Bash)을 열고 같은 폴더로 이동합니다. 오늘 터미널 명령어는 모두 이 폴더에서 입력합니다.

```bash
cd ~/security-agent-toolkit/agent_core
ls
```

목록에 이 노트북 파일이 보이면 됩니다.

### 0.2 오늘 오전의 순서

| 교시 | 무엇 |
|---|---|
| 2교시 | 폴링 ↔ 웹훅 · `Flask` 로 서버 만들기 |
| 3교시 | `curl` 로 요청 보내기 · `test_webhook.sh` |
| 4교시 | `argparse` · `webhook_server.py` 조립 |

### 0.3 오늘은 서버를 실행합니다

서버는 **끝나지 않는 프로그램**입니다. 셀에서 그냥 실행하면 그 셀이 끝나지 않아 다음 셀을 실행할 수 없습니다.
그래서 **서버를 백그라운드로 실행하는 함수** `start_server` 를 아래 셀에 둡니다. 내용은 몰라도 됩니다 — 실행만 하세요.

- 아래 셀은 함수를 **만들어 두기만** 합니다. 실행해도 서버는 아직 실행되지 않습니다.
- **2교시 실습을 시작할 때 한 번** 실행합니다. 그래야 뒤의 셀에서 `start_server("…")` 로 서버를 실행할 수 있습니다.
- 노트북을 다시 열었거나 `NameError: name 'start_server' is not defined` 가 나오면 아래 셀을 다시 실행합니다.
- 함께 들어 있는 `save_script` 는 3교시에 셸 스크립트 파일을 저장할 때 씁니다.

### 0.4 명령어는 터미널에 입력합니다

`curl` · `bash` · `cat` · `python 파일이름` 같은 명령어는 노트북 셀이 아니라 **터미널(Git Bash)** 에 입력합니다.
💻 표시가 있는 칸이 터미널에 입력할 명령어입니다.

In [ ]:
import subprocess
import sys
import time


def start_server(path, port=None):
    """서버 파일을 백그라운드로 실행합니다. 셀이 멈추지 않습니다."""
    command = [sys.executable, path]
    if port:
        command = command + ["--port", str(port)]
    proc = subprocess.Popen(command)
    time.sleep(2)
    print(f"{path} 를 실행했습니다")
    return proc


def save_script(path, text):
    """셸 스크립트를 파일로 저장합니다. 윈도우에서도 bash 가 읽을 수 있게 줄 끝을 맞춥니다."""
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        f.write(text.lstrip("\n"))
    print(f"{path} 를 저장했습니다")


---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">2교시 (10:00–10:50) · 받는 쪽이 된다</mark>


## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다. 검색이든 공식 문서든 AI 든 상관없습니다. 찾은 것은 노트북 셀이나 터미널에서 한 번 실행해 확인합니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **폴링(polling)** | 일정 간격으로 계속 물어보는 방식의 약점은 무엇인가 |
| **웹훅(webhook)** | 폴링과 견주면 무엇이 다른가 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다. 한 줄씩이면 됩니다.

- 폴링(polling) →
- 웹훅(webhook) →

적은 뒤 `Ctrl+S` 로 저장하면 노트북 파일에 함께 남습니다.

적어 둔 것은 이 교시가 끝날 때 다시 봅니다. 찾은 것과 배운 것이 어디서 갈렸는지가 오늘의 공부입니다.


---

## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">1 · 웹훅 수신 서버를 만든다</mark>


### 왜 필요한가

1. 지금까지는 **내가 필요할 때 물어보는** 방식이었습니다. 파일을 열고, API 를 부르고.
2. 그런데 침해 시도는 **내가 물어보는 시각에 맞춰** 일어나지 않습니다.
3. 방법이 둘입니다. 십 분마다 계속 물어보거나(**폴링**), 일이 생기면 상대가 알려 주게 하거나(**웹훅**).
4. 웹훅을 받으려면 **이쪽도 서버**가 되어야 합니다.


### 이 시간에 나오는 말

| 말 | 뜻 |
|---|---|
| 폴링 | 일정 간격으로 계속 물어보는 방식 |
| 웹훅 | 일이 생기면 상대가 먼저 알려 주는 방식 |
| 서버 | 요청을 기다렸다가 답하는 프로그램 |
| `Flask` | 파이썬으로 작은 서버를 만드는 패키지 |
| 라우트 | 어느 주소로 온 요청을 어느 함수가 받을지 정한 것 |


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">2.1 폴링 — 계속 물어본다</mark>

폴링은 **새 일이 없어도 계속 물어봅니다.** 그 낭비를 눈으로 봅니다.

```python
inbox = ["", "", "경보!", "", ""]      # 다섯 번 열어 보는데 한 번만 있다

for box in inbox:
    if box:
        print("일이 있다:", box)
    else:
        print("비어 있다")
```

다섯 번 물어서 한 번 건졌습니다. **네 번은 낭비**입니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 실행해 맞춰 봅니다.

```python
inbox = ["", "", "경보!", "", ""]

empty = 0
for box in inbox:
    if box:
        print("일이 있다:", box)
    else:
        empty = empty + 1

print("헛걸음", empty, "번")
```

막히면 바로 위 `2.1 폴링 — 계속 물어본다` 설명을 다시 봅니다.


In [ ]:
inbox = ["", "", "경보!", "", ""]

empty = 0
for box in inbox:
    if box:
        print("일이 있다:", box)
    else:
        empty = empty + 1

print("헛걸음", empty, "번")


✅ `일이 있다: 경보! · 헛걸음 4 번`


**빨리 알려면 자주 물어야 하고, 자주 물으면 헛걸음이 늘어납니다.** 웹훅은 이 맞바꿈이 없습니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-2 · 폴링 한 바퀴의 값 세기</font></h3></td></tr></table>

새 일이 **몇 번째**에 있었는지와 **헛걸음 횟수**를 함께 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `3번째에 있었습니다 · 헛걸음 2번` |

**💡 힌트**

1. 반복 안에서 먼저 1을 더하고 그다음 검사합니다.
2. 빈 글자 `""` 는 `if` 에서 거짓입니다. 값이 있으면 참입니다.
3. 찾으면 그 자리에서 출력합니다.

In [ ]:
inbox = ["", "", "경보!", "", ""]
turn = 0
empty = 0

for box in inbox:
    # 1. turn 에 1 을 더하세요

    # 2. box 에 값이 있으면 f"{turn}번째에 있었습니다 · 헛걸음 {empty}번" 을 출력하세요

    # 3. 값이 없으면 empty 에 1 을 더하세요

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-3 · 헛걸음 비율</font></h3></td></tr></table>

열 번 물어서 한 번만 건졌을 때 **헛걸음이 몇 퍼센트**인지 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `헛걸음 90.0%` |

**💡 힌트**

1. 전체 횟수와 건진 횟수를 이름 두 개에 담습니다.
2. 비율은 `헛걸음 / 전체 * 100` 입니다.
3. f-string 안에 그대로 넣어도 됩니다.


In [1]:
total = 10
hit = 1
total = 10
success = 1

fail = total - success
rate = fail / total * 100

print(f"헛걸음 {rate}%")

헛걸음 90.0%


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-4 · 간격을 바꿔 보기</font></h3></td></tr></table>

물어보는 횟수를 `5`·`10`·`60` 으로 바꿔 가며 **헛걸음 횟수**를 각각 출력하시오. 새 일은 언제나 한 번뿐입니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `5번 물으면 헛걸음 4번` · `10번 …9번` · `60번 …59번` |

**💡 힌트**

1. 횟수 셋을 리스트에 담고 `for` 로 돕니다.
2. 헛걸음은 전체에서 1을 뺀 값입니다.
3. 자주 물을수록 헛걸음이 늘어나는 것을 눈으로 봅니다.


In [2]:
for questions in [5, 10, 60]:
    failures = questions - 1
    print(f"{questions}번 물으면 헛걸음 {failures}번")# 여기에 코드를 입력하세요


5번 물으면 헛걸음 4번
10번 물으면 헛걸음 9번
60번 물으면 헛걸음 59번


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 2-5 · 언제 폴링이 낫나</font></h3></td></tr></table>

폴링이 웹훅보다 나은 경우를 **한 가지** 찾아 기록 셀에 적으시오. 코드가 아니라 글로 답하는 문제입니다.

- 힌트가 되는 질문 — 상대가 알려 줄 수 없는 상황은 언제인가?

| | |
|---|---|
| 🎯 확인 | 아래 셀에 주석으로 한 줄 적으면 됩니다 |

**💡 힌트**

1. 상대 쪽이 웹훅을 지원하지 않으면 어떻게 하나요?
2. 내 서버가 밖에서 보이지 않는 망 안에 있으면요?
3. 찾은 것을 한 줄로 적습니다. 정답은 하나가 아닙니다.


In [ ]:
상대 서버가 웹훅을 지원하지 않는 경우에는 폴링이 낫다.

### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">2.2 `Flask` — 서버를 만든다</mark>

서버는 **요청을 기다렸다가 답하는 프로그램**입니다. `Flask` 로 열 줄이면 만듭니다.

```python
from flask import Flask, request

app = Flask(__name__)


@app.route("/webhook", methods=["POST"])     # 이 주소로 POST 가 오면
def webhook():                                # 이 함수가 받는다
    event = request.get_json()                # 본문을 딕셔너리로
    return {"status": "ok"}, 200              # 답과 상태코드를 돌려준다


app.run(port=5000)
```

- `@app.route(...)` 한 줄이 **주소와 함수를 잇습니다.** 이것을 **라우트**라고 합니다.
- 앞에 붙은 `@` 는 오늘 처음 나옵니다. **뜻은 나중에 배우고, 오늘은 이 모양 그대로** 씁니다.
- 돌려주는 `200` 이 9/29에 배운 그 상태코드입니다. **받았다고 알리는 것**입니다.
- `app.run()` 은 **끝나지 않습니다.** 그래서 파일로 만들어 백그라운드로 실행합니다.


아래 셀로 서버 파일을 만듭니다. `Writing hello_server.py` 가 나오면 된 것입니다.


In [ ]:
%%writefile hello_server.py
from flask import Flask, request

app = Flask(__name__)


@app.route("/webhook", methods=["POST"])
def webhook():
    event = request.get_json()
    if "rule" in event:
        print("[수신]", event["rule"])
    else:
        print("[수신] rule 칸이 없는 요청")
    return {"status": "ok"}, 200


app.run(port=5001)


이제 도우미로 실행합니다. **5001 번** 포트를 씁니다.


In [ ]:
import subprocess
import sys
import time

def start_server(path, port=None):
    """서버 파일을 백그라운드로 실행합니다. 셀이 멈추지 않습니다."""
    command = [sys.executable, path]
    if port:
        command = command + ["--port", str(port)]
    proc = subprocess.Popen(command)
    time.sleep(2)
    print(f"{path} 를 실행했습니다")
    return proc

server1 = start_server("hello_server.py")   # 포트가 파일 안에 5001 로 적혀 있습니다


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-6 · 무엇이 보일까요</font></h3></td></tr></table>

이번에는 **GET** 으로 보냅니다. 서버는 POST 만 받기로 했습니다.

```bash
curl -s -o /dev/null -w "%{http_code}" http://127.0.0.1:5001/webhook
```

막히면 바로 위 `2.2 Flask — 서버를 만든다` 설명을 다시 봅니다.


💻 **터미널에 입력합니다** — 노트북 셀이 아닙니다.

```bash
curl -s -o /dev/null -w "%{http_code}" http://127.0.0.1:5001/webhook
```

✅ `405 — 그 주소는 있지만 그 메서드는 안 받는다는 뜻`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-7 · 받은 내용을 되돌려 주기</font></h3></td></tr></table>

서버가 받은 `rule` 값을 **응답에 담아** 돌려주게 만드시오.

1. 아래 셀의 번호 주석 아래에 코드를 쓰고 실행합니다. `echo_server.py` 파일이 만들어집니다.
2. 그 아래 **확인 셀**을 실행해 서버를 실행하고, 이어서 **확인** 명령을 터미널에 입력해 요청을 보냅니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `{"rule":"brute_force","status":"ok"}` |

**💡 힌트**

1. `event` 는 딕셔너리입니다. 칸이 있는지는 `if "rule" in event:` 로 확인합니다.
2. 돌려주는 딕셔너리에 `"rule": event["rule"]` 칸을 더합니다.
3. 포트는 파일 안에 5002 로 적혀 있습니다. 앞에서 실행한 서버(5001)와 겹치지 않습니다.

In [8]:
%%writefile echo_server.py
from flask import Flask, request

app = Flask(__name__)


@app.route("/webhook", methods=["POST"])
def webhook():
    event = request.get_json()
    # 1. event 에 "rule" 칸이 있으면 {"status": "ok", "rule": event 의 rule 값} 과 200 을 return 하세요

    # 2. 없으면 {"status": "ok"} 와 200 을 return 하세요



app.run(port=5002)

Writing echo_server.py


In [ ]:
# 1
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route("/echo", methods=["POST"])
def echo():
    data = request.json
    return jsonify({
        "rule": data["rule"],
        "status": "ok"
    })

app.run(port=5000)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit


💻 **확인 — 터미널에 그대로 입력합니다.**

```bash
curl -s -X POST -H "Content-Type: application/json" -d '{"rule":"brute_force"}' http://127.0.0.1:5002/webhook
```

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-8 · 받은 건수 세기</font></h3></td></tr></table>

서버가 **몇 건 받았는지** 세어 응답에 담으시오.

1. 아래 셀의 번호 주석 아래에 코드를 쓰고 실행합니다.
2. 그 아래 **확인 셀**을 실행해 서버를 실행하고, 이어서 **확인** 명령을 터미널에 입력해 요청을 두 번 보냅니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `{"count":1,"status":"ok"}` 다음에 `{"count":2,"status":"ok"}` |

**💡 힌트**

1. 받은 것을 쌓는 리스트는 함수 **밖에** 둡니다. 안에 두면 요청이 올 때마다 빈 리스트로 돌아갑니다.
2. 리스트에 더하는 것은 `.append()` 입니다.
3. 몇 건인지는 `len()` 으로 셉니다.

In [10]:
%%writefile count_server.py
from flask import Flask, request

app = Flask(__name__)
# 1. received 라는 변수를 만들어 빈 리스트를 담으세요


@app.route("/webhook", methods=["POST"])
def webhook():
    event = request.get_json()
    # 2. received 에 event 를 더하세요

    # 3. {"status": "ok", "count": received 의 길이} 와 200 을 return 하세요



app.run(port=5004)

Writing count_server.py


In [ ]:
# 확인 셀 — 그대로 실행합니다
server3 = start_server("count_server.py")   # 포트가 파일 안에 5004 로 적혀 있습니다

💻 **확인 — 터미널에 그대로 입력합니다.**

```bash
curl -s -X POST -H "Content-Type: application/json" -d '{"rule":"brute_force"}' http://127.0.0.1:5004/webhook
curl -s -X POST -H "Content-Type: application/json" -d '{"rule":"night_login"}' http://127.0.0.1:5004/webhook
```

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-9 · 받은 것을 파일로 남기기</font></h3></td></tr></table>

서버가 받은 alert(경보)를 **`received_alerts.json`** 에 쌓으시오. 9/28에 배운 `json.dump` 를 씁니다.

1. 아래 셀의 번호 주석 아래에 코드를 쓰고 실행합니다.
2. 그 아래 **확인 셀**을 실행해 서버를 실행하고, 이어서 **확인** 명령을 터미널에 입력해 요청을 두 번 보내고 파일을 열어 봅니다.

| | |
|---|---|
| 🎯 확인 | `received_alerts.json` 에 `brute_force` 와 `night_login` 두 건이 보인다 |

**💡 힌트**

1. 쓰기로 열 때는 `"w"` 와 `encoding="utf-8"` 을 넣습니다.
2. 받을 때마다 리스트를 통째로 다시 저장합니다.
3. `ensure_ascii=False` 와 `indent=2` 를 넣습니다.

In [ ]:
%%writefile save_server.py
import json
from flask import Flask, request

app = Flask(__name__)
received = []


@app.route("/webhook", methods=["POST"])
def webhook():
    received.append(request.get_json())
    # 1. "received_alerts.json" 을 쓰기로 여세요. 변수 이름은 f 입니다 (with 문)

        # 2. received 를 f 에 JSON 으로 저장하세요. 한글 그대로, 두 칸 들여쓰기입니다

    return {"status": "ok", "count": len(received)}, 200


app.run(port=5006)

In [ ]:
# 확인 셀 — 그대로 실행합니다
server4 = start_server("save_server.py")   # 포트가 파일 안에 5006 으로 적혀 있습니다

💻 **확인 — 터미널에 그대로 입력합니다.**

```bash
curl -s -X POST -H "Content-Type: application/json" -d '{"rule":"brute_force"}' http://127.0.0.1:5006/webhook
curl -s -X POST -H "Content-Type: application/json" -d '{"rule":"night_login"}' http://127.0.0.1:5006/webhook
cat received_alerts.json
```

<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 2-10 · 서버가 꺼지면 어떻게 되나</font></h3></td></tr></table>

실행 중인 서버를 **종료한 뒤** 같은 요청을 다시 보내시오. 이번에는 `curl` 에서 **`-s` 를 빼고** 보냅니다.

- 서버를 종료하는 법 — 아래 셀에서 `server1.terminate()`
- `curl` 명령은 터미널에 입력합니다.
- `-s` 는 진행 표시와 **오류 메시지**를 숨깁니다. `-s` 를 붙이면 아무것도 출력되지 않습니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `curl: (7) Failed to connect to 127.0.0.1 port 5001 …` |

⚠ **확인한 뒤 서버를 다시 실행합니다.** 3교시에도 5001 번 서버를 사용합니다.

```python
server1 = start_server("hello_server.py")
```

**💡 힌트**

1. `start_server` 가 돌려준 값을 이름에 담아 뒀습니다.
2. `.terminate()` 로 끕니다.
3. 종료한 뒤 `-s` 없이 `curl` 을 보냅니다. 파이썬 오류가 아니라 `curl` 의 **연결 실패 메시지**입니다.


In [ ]:
# 여기에 코드를 입력하세요


### <mark style="display:block; background:#bbdefb; color:#1a1a1a; padding:6px 12px; border-radius:4px">2.정리 📋 한눈에</mark>

| 쓰는 법 | 뜻 |
|---|---|
| 폴링 | 계속 물어본다. 빨리 알려면 헛걸음이 는다 |
| 웹훅 | 생기면 상대가 알려 준다. **받을 문**이 있어야 한다 |
| `@app.route("/webhook", methods=["POST"])` | 그 주소로 온 POST 를 이 함수가 받는다 |
| `request.get_json()` | 본문을 딕셔너리로 |
| `return {...}, 200` | 답과 상태코드를 돌려준다 |
| `app.run()` | **끝나지 않는다.** 백그라운드로 실행한다 |


---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">3교시 (11:00–11:50) · 손으로 두드린다</mark>


## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다. 검색이든 공식 문서든 AI 든 상관없습니다. 찾은 것은 노트북 셀이나 터미널에서 한 번 실행해 확인합니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **`curl`** | 터미널에서 요청을 보내는 명령. 무엇에 쓰나 |
| **GUI 와 CLI** | 마우스로 누르는 것과 글자로 적는 것은 무엇이 다른가 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다. 한 줄씩이면 됩니다.

- `curl` →
- GUI 와 CLI →

적은 뒤 `Ctrl+S` 로 저장하면 노트북 파일에 함께 남습니다.

적어 둔 것은 이 교시가 끝날 때 다시 봅니다. 찾은 것과 배운 것이 어디서 갈렸는지가 오늘의 공부입니다.


---

## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">2 · 서버가 도는지 확인한다</mark>


### 왜 필요한가

1. 서버를 실행했는데 **제대로 도는지** 알 방법이 있어야 합니다.
2. 확인하려고 또 파이썬 파일을 만드는 것은 번거롭습니다. **터미널에서 곧바로** 보냅니다.
3. 그 명령이 `curl` 입니다. 9/29에 배운 **메서드·헤더·본문** 세 조각이 옵션 셋으로 그대로 옵니다.


### 이 시간에 나오는 말

| 말 | 뜻 |
|---|---|
| CLI | 글자로 명령을 넣어 쓰는 방식 |
| GUI | 마우스로 눌러 쓰는 방식 |
| `curl` | 터미널에서 요청을 보내는 명령 |
| `-X` · `-H` · `-d` | 메서드 · 헤더 · 본문 |


### 손에 익혀 둘 명령 넷

터미널에서 쓰는 명령입니다. `cd` · `ls` 는 9/28·9/29 아침에 이미 쳐 봤습니다.

| 명령 | 무엇을 | 예 |
|---|---|---|
| `cd` | 폴더를 옮긴다 | `cd agent_core` · `cd ..` 는 한 칸 위로 |
| `ls` | 지금 폴더에 무엇이 있는지 본다 | `ls` · `ls -a` |
| `cat` | 파일 내용을 화면에 펼친다 | `cat agent.log` |
| `grep` | 특정 낱말이 든 줄만 찾는다 | `grep ERROR agent.log` |

`cat` · `grep` 은 **오늘은 눈으로만 봅니다.** 직접 쳐 보는 것은 10/12 아침입니다.


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">3.1 `curl` — 세 조각을 옵션으로</mark>

| 옵션 | 무엇 | 9/29에 배운 것 |
|---|---|---|
| `-X POST` | 메서드 | `GET` / `POST` |
| `-H "Content-Type: application/json"` | 헤더 | 요청에 덧붙이는 정보 |
| `-d '{"rule":"…"}'` | 본문 | 보낼 내용 |
| `-s` | 진행 막대를 숨긴다 | — |

```bash
curl -s -X POST -H "Content-Type: application/json" \
  -d '{"rule":"brute_force"}' http://127.0.0.1:5001/webhook
```

이 명령은 노트북 셀이 아니라 **터미널(Git Bash)** 에 입력합니다. 줄 끝의 `\` 는 명령이 다음 줄에 이어진다는 표시입니다.

**2교시에 실행한 5001 번 서버를 계속 사용합니다.**
⭐도전 2-10 에서 서버를 종료했다면 `server1 = start_server("hello_server.py")` 로 다시 실행한 뒤 요청을 보냅니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-1 · 무엇이 보일까요</font></h3></td></tr></table>

이번에는 **헤더를 빼고** 보냅니다. 상태코드만 봅니다.

```bash
curl -s -o /dev/null -w "%{http_code}" -X POST -d '{"rule":"night_login"}' http://127.0.0.1:5001/webhook
```

막히면 바로 위 `3.1 curl — 세 조각을 옵션으로` 설명을 다시 봅니다.


💻 **터미널에 입력합니다** — 노트북 셀이 아닙니다.

```bash
curl -s -o /dev/null -w "%{http_code}" -X POST -d '{"rule":"night_login"}' http://127.0.0.1:5001/webhook
```

✅ `415 — 요청 본문 형식을 알리지 않아 서버가 거절했다`


`415` 는 **「이 형식은 처리할 수 없다」**는 뜻입니다. 헤더가 왜 필요한지가 여기서 드러납니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-2 · 다른 alert(경보) 보내기</font></h3></td></tr></table>

`rule` 을 `password_spraying` 으로 바꿔 보내시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `{"status":"ok"}` |

**💡 힌트**

1. 앞 명령에서 `-d` 뒤의 값만 바꿉니다.
2. 작은따옴표 안에 큰따옴표로 적습니다.
3. 주소는 그대로 5001 번입니다.


💻 **터미널에 입력합니다.** 입력한 명령은 이 셀을 두 번 눌러 아래 칸에 적어 둡니다.

```bash

```

In [ ]:
curl -X POST http://localhost:5000/alert \
-H "Content-Type: application/json" \
-d '{"rule":"password_spraying"}'

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-3 · 상태코드만 보기</font></h3></td></tr></table>

같은 요청을 보내되 **응답 본문은 버리고 상태코드만** 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `200` |

**💡 힌트**

1. 본문을 버리는 옵션은 `-o /dev/null` 입니다.
2. 상태코드를 출력하는 옵션은 `-w "%{http_code}"` 입니다.
3. 문제 3-1의 명령에서 헤더만 되살리면 됩니다.


💻 **터미널에 입력합니다.** 입력한 명령은 이 셀을 두 번 눌러 아래 칸에 적어 둡니다.

```bash

```

In [ ]:
response = requests.get(url)
print(response.status_code)

response = requests.get(url + "/none")
print(response.status_code)

**💡 힌트**

1. 주소 끝만 `/webhook` 에서 `/none` 으로 바꿉니다.
2. 상태코드만 보는 옵션은 문제 3-3와 같습니다.
3. `404` 는 9/29에 배운 「그런 주소가 없다」입니다.


💻 **터미널에 입력합니다.** 입력한 명령은 이 셀을 두 번 눌러 아래 칸에 적어 둡니다.

```bash

```

<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 3-5 · 네 가지 상태코드를 모아 보기</font></h3></td></tr></table>

같은 서버에 네 가지로 보내 보고 **상태코드 넷**을 표로 정리하시오.

| 어떻게 보내나 | 무슨 코드가 오나 |
|---|---|
| 헤더·본문 갖춰 POST | |
| GET 으로 | |
| 없는 주소로 | |
| 헤더 없이 POST | |

| | |
|---|---|
| 🎯 나와야 하는 결과 | `200` · `405` · `404` · `415` |

**💡 힌트**

1. 네 번 따로 보내고 코드만 봅니다.
2. `405` 는 「그 주소는 있는데 그 방법은 안 받는다」입니다.
3. 코드마다 **누구 잘못인지**를 옆에 적어 봅니다 — 9/29의 앞자리 이야기입니다.


💻 **터미널에 입력합니다.** 입력한 명령은 이 셀을 두 번 눌러 아래 칸에 적어 둡니다.

```bash

```

### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">3.2 `test_webhook.sh` — 확인을 파일로 묶는다</mark>

alert(경보)가 세 가지면 `curl` 을 세 번 칩니다. 매번 손으로 치면 오타가 납니다.
**한 파일에 묶어 두고 한 줄로 부릅니다.**

```python
save_script("test_webhook.sh", r"""
curl -s -X POST -H "Content-Type: application/json" \
  -d '{"rule":"brute_force"}' http://127.0.0.1:5001/webhook
echo
curl -s -X POST -H "Content-Type: application/json" \
  -d '{"rule":"night_login"}' http://127.0.0.1:5001/webhook
echo
""")
```

- `echo` 는 **줄을 바꿔 주는 것**뿐입니다. 없으면 답이 한 줄에 붙습니다.
- `save_script("파일이름", r"""…""")` 는 따옴표 세 개 사이의 내용을 그 파일로 저장합니다. 맨 위 준비 셀에 있는 함수입니다.
- 실행은 터미널에서 `bash test_webhook.sh` 입니다.
- 셸 스크립트 문법은 오늘 배우지 않습니다. **`curl` 을 줄줄이 적어 둔 파일**로만 씁니다.


In [ ]:
save_script("demo_test.sh", r"""
curl -s -X POST -H "Content-Type: application/json" \
  -d '{"rule":"brute_force"}' http://127.0.0.1:5001/webhook
echo
curl -s -X POST -H "Content-Type: application/json" \
  -d '{"rule":"night_login"}' http://127.0.0.1:5001/webhook
echo
""")

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-6 · 무엇이 보일까요</font></h3></td></tr></table>

이번에는 실행합니다.

```bash
bash demo_test.sh
```

막히면 바로 위 `3.2 test_webhook.sh — 확인을 파일로 묶는다` 설명을 다시 봅니다.


💻 **터미널에 입력합니다** — 노트북 셀이 아닙니다.

```bash
bash demo_test.sh
```

✅ `{"status":"ok"}` 가 두 번 출력되고, 그 사이에 `echo` 가 출력한 빈 줄이 있다


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-7 · `test_webhook.sh` 만들기</font></h3></td></tr></table>

오늘의 산출물 **`test_webhook.sh`** 를 만드시오. alert(경보) **세 가지**를 보냅니다.

- `brute_force` · `password_spraying` · `night_login` — 9/29에 만든 룰 세 개입니다.
- 앞의 두 가지는 미리 적혀 있습니다. **셋째 요청**을 같은 모양으로 적습니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 다음 문제에서 실행하면 `{"status":"ok"}` 세 줄 |

**💡 힌트**

1. 위의 `curl` 한 덩이와 같은 모양으로 적고 `rule` 값만 바꿉니다.
2. 첫 줄 끝의 `\` 뒤에는 빈칸이 없어야 합니다.
3. 덩이 끝에 `echo` 를 둡니다. 없으면 답이 한 줄에 붙습니다.

In [ ]:
save_script("test_webhook.sh", r"""
curl -s -X POST -H "Content-Type: application/json" \
  -d '{"rule":"brute_force"}' http://127.0.0.1:5001/webhook
echo
curl -s -X POST -H "Content-Type: application/json" \
  -d '{"rule":"password_spraying"}' http://127.0.0.1:5001/webhook
echo
# 1. 위와 같은 모양으로 rule 이 "night_login" 인 요청을 적으세요 (curl 명령과 echo)
""")

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-8 · 돌려서 확인하기</font></h3></td></tr></table>

만든 파일을 실행하고, 응답이 **세 줄** 나오는지 보시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `{"status":"ok"}` 세 줄 |

**💡 힌트**

1. 실행은 터미널에서 `bash 파일이름` 입니다.
2. 터미널이 `agent_core` 폴더에 있어야 파일을 찾습니다.
3. 답이 한 줄에 붙으면 `echo` 가 빠진 것입니다.


💻 **터미널에 입력합니다.** 입력한 명령은 이 셀을 두 번 눌러 아래 칸에 적어 둡니다.

```bash

```

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-9 · 보낸 alert(경보) 이름도 함께 출력하기</font></h3></td></tr></table>

어느 alert 를 보내는 중인지 **화면에 함께** 나오게 `test_webhook.sh` 를 고치시오.

- 셸에서 글자를 출력하는 명령도 `echo` 입니다. `-n` 을 붙이면 줄을 바꾸지 않습니다.
- 아래 셀의 번호 주석 아래에 한 줄씩 쓰고 실행한 뒤, 그 아래 **확인** 명령을 터미널에 입력합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `brute_force 보냄 {"status":"ok"}` 처럼 세 줄 |

**💡 힌트**

1. 첫째 줄은 `echo -n "brute_force 보냄 "` 입니다.
2. 나머지 둘도 이름만 바꿔 같은 모양으로 씁니다.
3. `-n` 을 빼면 이름과 답이 서로 다른 줄에 나옵니다.

In [ ]:
save_script("test_webhook.sh", r"""
# 1. "brute_force 보냄 " 을 줄을 바꾸지 않고 출력하세요

curl -s -X POST -H "Content-Type: application/json" \
  -d '{"rule":"brute_force"}' http://127.0.0.1:5001/webhook
echo
# 2. "password_spraying 보냄 " 을 줄을 바꾸지 않고 출력하세요

curl -s -X POST -H "Content-Type: application/json" \
  -d '{"rule":"password_spraying"}' http://127.0.0.1:5001/webhook
echo
# 3. "night_login 보냄 " 을 줄을 바꾸지 않고 출력하세요

curl -s -X POST -H "Content-Type: application/json" \
  -d '{"rule":"night_login"}' http://127.0.0.1:5001/webhook
echo
""")

💻 **확인 — 터미널에 그대로 입력합니다.**

```bash
bash test_webhook.sh
```

<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 3-10 · 상태코드까지 출력하게 하기</font></h3></td></tr></table>

`test_webhook.sh` 가 **응답 본문 대신 상태코드**를 출력하게 고치시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `brute_force 200` 꼴로 세 줄 |

**💡 힌트**

1. 문제 3-3의 `-o /dev/null -w "%{http_code}"` 를 씁니다.
2. `echo -n` 으로 이름을 먼저 출력합니다.
3. 코드만 보면 **성공·실패를 한눈에** 훑을 수 있습니다.


In [ ]:
# 여기에 코드를 입력하세요


### <mark style="display:block; background:#bbdefb; color:#1a1a1a; padding:6px 12px; border-radius:4px">3.정리 📋 한눈에</mark>

| 쓰는 법 | 뜻 |
|---|---|
| `curl -X POST` | 메서드를 정한다 |
| `-H "Content-Type: application/json"` | 요청 본문 형식을 알린다. 빼면 **415** |
| `-d '{...}'` | 본문을 실어 보낸다 |
| `-o /dev/null -w "%{http_code}"` | 본문은 버리고 상태코드만 |
| `bash 파일.sh` | `curl` 을 묶어 둔 파일을 한 번에 돌린다 |

| 코드 | 언제 |
|---|---|
| `200` | 갖춰 보냈다 |
| `404` | 그런 주소가 없다 |
| `405` | 주소는 있는데 메서드가 다르다 |
| `415` | 요청 본문 형식을 처리할 수 없다 |


---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">4교시 (12:00–12:50) · 설정을 밖으로</mark>


## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다. 검색이든 공식 문서든 AI 든 상관없습니다. 찾은 것은 노트북 셀이나 터미널에서 한 번 실행해 확인합니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **`argparse`** | 명령 뒤에 붙이는 값을 어떻게 받나 |
| **기본값(default)** | 인자를 안 주면 어떻게 되게 하나 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다. 한 줄씩이면 됩니다.

- `argparse` →
- 기본값(default) →

적은 뒤 `Ctrl+S` 로 저장하면 노트북 파일에 함께 남습니다.

적어 둔 것은 이 교시가 끝날 때 다시 봅니다. 찾은 것과 배운 것이 어디서 갈렸는지가 오늘의 공부입니다.


---

## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">3 · 포트를 코드 밖으로</mark>


### 왜 필요한가

1. 지금 서버는 포트가 **코드에 적혀** 있습니다. 바꾸려면 파일을 고쳐야 합니다.
2. 같은 서버를 두 개 실행하려면 포트가 달라야 하는데, 그때마다 파일을 고칠 수는 없습니다.
3. **명령 뒤에 붙여** 넘기면 됩니다 — `python webhook_server.py --port 5001`.
4. 9/30에 `.env` 로 **비밀 값**을 밖에 뺐습니다. 오늘은 **설정 값**을 뺍니다. 같은 생각입니다.


### 이 시간에 나오는 말

| 말 | 뜻 |
|---|---|
| CLI 인자 | 명령 뒤에 붙여 넘기는 값 |
| `argparse` | 그 값을 받아 주는 파이썬 기본 도구 |
| `--port` | 이름을 붙인 인자. 순서를 안 외워도 된다 |
| 기본값 | 안 줬을 때 대신 쓰는 값 |


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.1 `argparse` — 명령 뒤의 값을 받는다</mark>

```python
import argparse

parser = argparse.ArgumentParser(description="경보를 받는 웹훅 서버")
parser.add_argument("--port", type=int, default=5000, help="열어 둘 포트 번호")
args = parser.parse_args()

print(args.port)
```

- `type=int` 를 빼면 **글자로** 들어옵니다. 숫자로 쓰려면 적어야 합니다.
- `default=5000` 덕분에 **안 줘도 돕니다.**
- `--help` 를 붙이면 **적어 둔 설명이 그대로** 나옵니다. 공짜로 얻는 사용법입니다.


In [ ]:
%%writefile port_demo.py
import argparse

parser = argparse.ArgumentParser(description="포트를 받아 출력해 봅니다")
parser.add_argument("--port", type=int, default=5000, help="열어 둘 포트 번호")
args = parser.parse_args()

print("받은 포트:", args.port)


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-1 · 무엇이 보일까요</font></h3></td></tr></table>

인자를 주지 않고 실행합니다.

```bash
python port_demo.py
```

막히면 바로 위 `4.1 argparse — 명령 뒤의 값을 받는다` 설명을 다시 봅니다.


💻 **터미널에 입력합니다** — 노트북 셀이 아닙니다.

```bash
python port_demo.py
```

✅ `받은 포트: 5000`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-2 · 인자를 주고 실행하기</font></h3></td></tr></table>

`port_demo.py` 에 **5001** 을 넘겨 실행하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `받은 포트: 5001` |

**💡 힌트**

1. 인자는 파일 이름 뒤에 붙입니다.
2. 이름을 붙인 인자라 `--port` 를 함께 적습니다.
3. 값은 빈칸 하나를 두고 적습니다.


💻 **터미널에 입력합니다.** 입력한 명령은 이 셀을 두 번 눌러 아래 칸에 적어 둡니다.

```bash

```

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-3 · 인자를 하나 더 받기</font></h3></td></tr></table>

**`--rule`** 인자를 더 받아 함께 출력하는 파일을 만드시오.
`--rule` 을 적지 않고 실행하면 `brute_force` 를 사용하게 합니다(`--rule` 의 기본값).

| | |
|---|---|
| 🎯 나와야 하는 결과 | 확인 명령을 터미널에 입력하면 `포트 5000 · 룰 brute_force` |

**💡 힌트**

1. `add_argument` 를 한 줄 더 씁니다. 위의 `--port` 줄과 같은 모양입니다.
2. 글자라서 `type=int` 는 안 붙입니다.
3. 받은 값은 `args.rule` 로 꺼냅니다.

In [ ]:
%%writefile rule_demo.py
import argparse

parser = argparse.ArgumentParser(description="포트와 룰을 받습니다")
parser.add_argument("--port", type=int, default=5000, help="열어 둘 포트 번호")
# 1. --rule 인자를 더 받으세요. --rule 의 기본값은 "brute_force", 설명은 "보낼 룰 이름" 입니다

args = parser.parse_args()

# 2. f-string 으로 "포트 5000 · 룰 brute_force" 꼴을 출력하세요. 두 값은 args 에서 꺼냅니다

💻 **확인 — 터미널에 그대로 입력합니다.**

```bash
python rule_demo.py
```

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-4 · `type=int` 를 빼 보기</font></h3></td></tr></table>

`--port` 에서 `type=int` 를 **빼고** 만든 뒤, 받은 값에 글자 `"1"` 을 더해 보시오. 무엇이 달라지는지 확인하는 문제입니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 확인 명령을 터미널에 입력하면 숫자가 아니라 **글자끼리 붙는다** — `50001` |

**💡 힌트**

1. `add_argument` 에 `type=int` 를 쓰지 않습니다.
2. `args.port + "1"` 처럼 글자를 더합니다.
3. 글자로 들어오면 `+` 가 이어 붙이기가 됩니다 — 9/28에 본 그 이야기입니다.

In [ ]:
%%writefile notype_demo.py
import argparse

parser = argparse.ArgumentParser()
# 1. --port 인자를 받으세요. type=int 를 쓰지 않고, 기본값은 글자 "5000" 입니다

args = parser.parse_args()

# 2. args.port 에 글자 "1" 을 더한 것을 출력하세요

💻 **확인 — 터미널에 그대로 입력합니다.**

```bash
python notype_demo.py
```

<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 4-5 · 없는 인자를 주면</font></h3></td></tr></table>

`--포트없음` 처럼 **정하지 않은 인자**를 주면 어떻게 되는지 확인하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `unrecognized arguments` 라는 오류와 함께 **사용법이 출력된다** |

**💡 힌트**

1. `python port_demo.py --wrong 1` 처럼 아무 이름이나 줍니다.
2. `argparse` 가 **대신 오류를 내 주고** 사용법까지 보여 줍니다.
3. 우리가 검사 코드를 안 써도 되는 이유입니다.


💻 **터미널에 입력합니다.** 입력한 명령은 이 셀을 두 번 눌러 아래 칸에 적어 둡니다.

```bash

```

### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.2 `webhook_server.py` 조립</mark>

오늘 만든 조각을 한 파일로 잇습니다. 새 문법은 없습니다.

| 조각 | 어디서 |
|---|---|
| `Flask` 라우트 | 2교시 |
| `argparse --port` | 이 교시 |
| 받은 것을 파일로 | 2교시 문제 2-9 |

이름은 학원 교안이 정한 **`webhook_server.py`** 그대로 씁니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-6 · 무엇이 보일까요</font></h3></td></tr></table>

이번에는 **인자를 주고** 돌립니다. 기본값이 아니라 준 값이 쓰입니다.

```bash
python port_demo.py --port 5002
```

막히면 바로 위 `4.2 webhook_server.py 조립` 설명을 다시 봅니다.


💻 **터미널에 입력합니다** — 노트북 셀이 아닙니다.

```bash
python port_demo.py --port 5002
```

✅ `받은 포트: 5002`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-7 · `webhook_server.py` 만들기</font></h3></td></tr></table>

오전의 산출물 **`webhook_server.py`** 를 만드시오. 문제 2-9 의 서버에 **`--port` 인자**를 더한 것입니다.

1. 서버의 나머지 줄은 미리 적혀 있습니다. 번호 주석 세 군데를 채웁니다.
2. 다 채운 뒤 실행하면 파일이 만들어집니다. 실행하는 것은 다음 문제에서 합니다.

| | |
|---|---|
| 🎯 확인 | 다음 문제에서 5003 번으로 실행하면 `count` 가 늘어난다 |

**💡 힌트**

1. `argparse` 세 줄은 문제 4-3 에서 쓴 것과 같은 모양입니다. `--port` 는 정수라서 `type=int` 를 붙입니다.
2. 돌려주는 줄은 문제 2-8 과 같습니다.
3. 마지막 줄은 `app.run(port=args.port)` 입니다. 받은 인자를 포트로 씁니다.

In [ ]:
%%writefile webhook_server.py
import argparse
import json
from flask import Flask, request

# 1. parser 를 만들고, --port 인자(정수, 기본값 5000)를 받고, args 에 parse_args() 결과를 담으세요


app = Flask(__name__)
received = []


@app.route("/webhook", methods=["POST"])
def webhook():
    event = request.get_json()
    received.append(event)
    with open("received_alerts.json", "w", encoding="utf-8") as f:
        json.dump(received, f, ensure_ascii=False, indent=2)
    if "rule" in event:
        print("[수신]", event["rule"])
    # 2. {"status": "ok", "count": received 의 길이} 와 200 을 return 하세요



# 3. args 로 받은 포트로 app 을 실행하세요

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-8 · 실행하고 요청을 보내 확인하기</font></h3></td></tr></table>

만든 서버를 **5003** 번에 실행하고, `curl` 로 요청을 두 번 보내 `count` 가 늘어나는지 보시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `{"count":1,"status":"ok"}` 다음에 `{"count":2,"status":"ok"}` |

서버는 아래 셀에서 실행하고, `curl` 명령은 터미널에 입력합니다.

**💡 힌트**

1. 실행하는 것은 맨 위 도우미 `start_server` 입니다.
2. `curl` 주소의 포트를 5003 으로 바꿉니다.
3. 두 번 보내면 숫자가 늘어납니다.


In [ ]:
# 여기에 코드를 입력하세요


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-9 · 깃허브에 올리기</font></h3></td></tr></table>

오전 산출물 두 개를 깃허브 저장소 `security-agent-toolkit` 에 올리시오.

| | |
|---|---|
| 🎯 확인 | 깃허브 저장소의 `agent_core` 폴더에 `webhook_server.py` 와 `test_webhook.sh` 가 보인다 |

**💡 힌트**

1. `git status` 로 새로 만든 파일이 목록에 있는지 먼저 봅니다.
2. `git add` 뒤에 올릴 파일 두 개를 적습니다.
3. `test_webhook.sh` 는 **3-7 에서 만들고 3-9 에서 고친** 그 파일입니다.

💻 **터미널에 입력합니다** — 노트북 셀이 아닙니다.

```bash
cd ~/security-agent-toolkit
git status
git add agent_core/webhook_server.py agent_core/test_webhook.sh
git commit -m "Add webhook server and test script"
git push
cd agent_core
```

<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 4-10 · 받은 것을 열어 보기</font></h3></td></tr></table>

서버가 남긴 **`received_alerts.json`** 을 읽어 **룰 이름만** 한 줄씩 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 보낸 alert(경보) 이름들이 한 줄씩 |

**💡 힌트**

1. 9/28에 배운 `json.load` 를 씁니다.
2. 읽어 온 것은 리스트입니다.
3. `for` 로 돌며 `row["rule"]` 을 출력합니다.


In [ ]:
# 여기에 코드를 입력하세요


### <mark style="display:block; background:#bbdefb; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.정리 📋 한눈에</mark>

| 쓰는 법 | 뜻 |
|---|---|
| `parser.add_argument("--port", type=int, default=5000)` | 이름 붙인 인자를 받는다 |
| `args.port` | 받은 값을 꺼낸다 |
| `--help` | 적어 둔 설명이 그대로 사용법이 된다 |
| `type=int` | 빼면 **글자로** 들어온다 |

오전의 산출물은 `agent_core` 폴더의 **`webhook_server.py`** 와 **`test_webhook.sh`** 입니다. 문제 4-9 에서 깃허브에 올립니다.

오후에는 이 서버로 **alert 를 자동으로 전송하는 프로그램**을 만듭니다.


---

# <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">정답 · 먼저 풀어 본 뒤에 엽니다</mark>

정답은 아래에 그대로 있습니다. 먼저 스스로 풀어 본 뒤에 봅니다.


In [ ]:
# 정답 2-3
total = 10
hit = 1

empty = total - hit

print(f"헛걸음 {empty / total * 100}%")


In [ ]:
# 정답 ⭐2-5
print("상대가 웹훅을 지원하지 않을 때")
print("내 서버가 바깥에서 닿지 않는 망 안에 있을 때")
print("놓친 것이 있는지 주기적으로 다시 훑어야 할 때")


In [ ]:
# 정답 2-7
code = r"""from flask import Flask, request

app = Flask(__name__)


@app.route("/webhook", methods=["POST"])
def webhook():
    event = request.get_json()
    # 1. event 에 "rule" 칸이 있으면 {"status": "ok", "rule": event 의 rule 값} 과 200 을 return 하세요
    if "rule" in event:
        return {"status": "ok", "rule": event["rule"]}, 200
    # 2. 없으면 {"status": "ok"} 와 200 을 return 하세요
    return {"status": "ok"}, 200


app.run(port=5002)
"""

with open("echo_server.py", "w", encoding="utf-8") as f:
    f.write(code)

print("echo_server.py 를 만들었습니다.")

In [ ]:
# 정답 2-8
code = r"""from flask import Flask, request

app = Flask(__name__)
# 1. received 라는 변수를 만들어 빈 리스트를 담으세요
received = []

@app.route("/webhook", methods=["POST"])
def webhook():
    event = request.get_json()
    # 2. received 에 event 를 더하세요
    received.append(event)
    # 3. {"status": "ok", "count": received 의 길이} 와 200 을 return 하세요
    return {"status": "ok", "count": len(received)}, 200


app.run(port=5004)
"""

with open("count_server.py", "w", encoding="utf-8") as f:
    f.write(code)

print("count_server.py 를 만들었습니다.")

In [ ]:
# 정답 2-9
code = r"""import json
from flask import Flask, request

app = Flask(__name__)
received = []


@app.route("/webhook", methods=["POST"])
def webhook():
    received.append(request.get_json())
    # 1. "received_alerts.json" 을 쓰기로 여세요. 변수 이름은 f 입니다 (with 문)
    with open("received_alerts.json", "w", encoding="utf-8") as f:
        # 2. received 를 f 에 JSON 으로 저장하세요. 한글 그대로, 두 칸 들여쓰기입니다
        json.dump(received, f, ensure_ascii=False, indent=2)
    return {"status": "ok", "count": len(received)}, 200


app.run(port=5006)
"""

with open("save_server.py", "w", encoding="utf-8") as f:
    f.write(code)

print("save_server.py 를 만들었습니다.")

In [ ]:
# 정답 ⭐2-10
print("server1.terminate()  # 서버를 종료한다")
print('터미널: curl -X POST -H "Content-Type: application/json" -d \'{"rule":"brute_force"}\' http://127.0.0.1:5001/webhook')
print()
print("curl: (7) Failed to connect to 127.0.0.1 port 5001 ... 이 출력됩니다.")
print("-s 를 붙이면 오류 메시지도 숨겨져 아무것도 출력되지 않습니다.")
print()
print('server1 = start_server("hello_server.py")  # 3교시에 사용하므로 다시 실행한다')
print("서버가 종료되어 있으면 클라이언트는 응답을 받지 못합니다. 경보가 전달되지 않습니다.")


In [ ]:
# 정답 3-2
print("아래 명령을 터미널에 입력합니다.")
print()
print('curl -s -X POST -H "Content-Type: application/json" \\')
print('  -d \'{"rule":"password_spraying"}\' http://127.0.0.1:5001/webhook')


In [ ]:
# 정답 3-3
print('curl -s -o /dev/null -w "%{http_code}" \\')
print('  -X POST -H "Content-Type: application/json" \\')
print('  -d \'{"rule":"brute_force"}\' http://127.0.0.1:5001/webhook')


In [ ]:
# 정답 3-4
print('curl -s -o /dev/null -w "%{http_code}" http://127.0.0.1:5001/none')


In [ ]:
# 정답 ⭐3-5
print("200  갖춰 보냄        — 성공")
print("405  GET 으로          — 주소는 있으나 메서드가 다르다")
print("404  /none 으로        — 그런 주소가 없다")
print("415  헤더 없이         — 요청 본문 형식을 처리할 수 없다")
print()
print("200 은 성공입니다. 나머지 셋은 앞자리가 4 입니다 — 요청을 보낸 쪽이 고쳐야 합니다.")


In [ ]:
# 정답 3-7
save_script("test_webhook.sh", r"""
curl -s -X POST -H "Content-Type: application/json" \
  -d '{"rule":"brute_force"}' http://127.0.0.1:5001/webhook
echo
curl -s -X POST -H "Content-Type: application/json" \
  -d '{"rule":"password_spraying"}' http://127.0.0.1:5001/webhook
echo
# 1. 위와 같은 모양으로 rule 이 "night_login" 인 요청을 적으세요 (curl 명령과 echo)
curl -s -X POST -H "Content-Type: application/json" \
  -d '{"rule":"night_login"}' http://127.0.0.1:5001/webhook
echo
""")


In [ ]:
# 정답 3-8
print("bash test_webhook.sh")


In [ ]:
# 정답 3-9
save_script("test_webhook.sh", r"""
# 1. "brute_force 보냄 " 을 줄을 바꾸지 않고 출력하세요
echo -n "brute_force 보냄 "
curl -s -X POST -H "Content-Type: application/json" \
  -d '{"rule":"brute_force"}' http://127.0.0.1:5001/webhook
echo
# 2. "password_spraying 보냄 " 을 줄을 바꾸지 않고 출력하세요
echo -n "password_spraying 보냄 "
curl -s -X POST -H "Content-Type: application/json" \
  -d '{"rule":"password_spraying"}' http://127.0.0.1:5001/webhook
echo
# 3. "night_login 보냄 " 을 줄을 바꾸지 않고 출력하세요
echo -n "night_login 보냄 "
curl -s -X POST -H "Content-Type: application/json" \
  -d '{"rule":"night_login"}' http://127.0.0.1:5001/webhook
echo
""")


In [ ]:
# 정답 ⭐3-10
lines = []
for rule in ["brute_force", "password_spraying", "night_login"]:
    lines.append(f'echo -n "{rule} "')
    lines.append('curl -s -o /dev/null -w "%{http_code}" \\')
    lines.append('  -X POST -H "Content-Type: application/json" \\')
    lines.append(f"  -d '{{\"rule\":\"{rule}\"}}' http://127.0.0.1:5001/webhook")
    lines.append("echo")

with open("test_webhook.sh", "w", encoding="utf-8", newline="\n") as f:
    f.write("\n".join(lines) + "\n")

print(open("test_webhook.sh", encoding="utf-8").read())


In [ ]:
# 정답 4-2
print("python port_demo.py --port 5001")


In [ ]:
# 정답 4-3
code = r"""import argparse

parser = argparse.ArgumentParser(description="포트와 룰을 받습니다")
parser.add_argument("--port", type=int, default=5000, help="열어 둘 포트 번호")
# 1. --rule 인자를 더 받으세요. --rule 의 기본값은 "brute_force", 설명은 "보낼 룰 이름" 입니다
parser.add_argument("--rule", default="brute_force", help="보낼 룰 이름")
args = parser.parse_args()

# 2. f-string 으로 "포트 5000 · 룰 brute_force" 꼴을 출력하세요. 두 값은 args 에서 꺼냅니다
print(f"포트 {args.port} · 룰 {args.rule}")
"""

with open("rule_demo.py", "w", encoding="utf-8") as f:
    f.write(code)

print("rule_demo.py 를 만들었습니다.")

In [ ]:
# 정답 4-4
code = r"""import argparse

parser = argparse.ArgumentParser()
# 1. --port 인자를 받으세요. type=int 를 쓰지 않고, 기본값은 글자 "5000" 입니다
parser.add_argument("--port", default="5000")
args = parser.parse_args()

# 2. args.port 에 글자 "1" 을 더한 것을 출력하세요
print(args.port + "1")
"""

with open("notype_demo.py", "w", encoding="utf-8") as f:
    f.write(code)

print("notype_demo.py 를 만들었습니다.")

In [ ]:
# 정답 ⭐4-5
print("python port_demo.py --wrong 1")
print()
print("argparse 가 대신 막아 줍니다 — 우리가 검사 코드를 쓰지 않아도 됩니다.")


In [ ]:
# 정답 4-7
code = r"""import argparse
import json
from flask import Flask, request

# 1. parser 를 만들고, --port 인자(정수, 기본값 5000)를 받고, args 에 parse_args() 결과를 담으세요
parser = argparse.ArgumentParser(description="경보를 받는 웹훅 서버")
parser.add_argument("--port", type=int, default=5000, help="열어 둘 포트 번호")
args = parser.parse_args()

app = Flask(__name__)
received = []


@app.route("/webhook", methods=["POST"])
def webhook():
    event = request.get_json()
    received.append(event)
    with open("received_alerts.json", "w", encoding="utf-8") as f:
        json.dump(received, f, ensure_ascii=False, indent=2)
    if "rule" in event:
        print("[수신]", event["rule"])
    # 2. {"status": "ok", "count": received 의 길이} 와 200 을 return 하세요
    return {"status": "ok", "count": len(received)}, 200


# 3. args 로 받은 포트로 app 을 실행하세요
app.run(port=args.port)
"""

with open("webhook_server.py", "w", encoding="utf-8") as f:
    f.write(code)

print("webhook_server.py 를 만들었습니다.")

In [ ]:
# 정답 4-8
print('server3 = start_server("webhook_server.py", 5003)')
print()
print('터미널에 입력합니다.')
print('curl -s -X POST -H "Content-Type: application/json" -d \'{"rule":"brute_force"}\' http://127.0.0.1:5003/webhook')
print('curl -s -X POST -H "Content-Type: application/json" -d \'{"rule":"night_login"}\' http://127.0.0.1:5003/webhook')


In [ ]:
# 정답 4-9
print("cd ~/security-agent-toolkit")
print("git status")
print("git add agent_core/webhook_server.py agent_core/test_webhook.sh")
print('git commit -m "Add webhook server and test script"')
print("git push")


In [ ]:
# 정답 ⭐4-10
import json
import os

path = "received_alerts.json"

if os.path.exists(path):
    with open(path, encoding="utf-8") as f:
        rows = json.load(f)
    for row in rows:
        print(row["rule"])
else:
    print("아직 받은 경보가 없습니다 — 서버를 실행하고 curl 로 요청을 보낸 뒤에 다시 실행하세요")
